# Member 3: Recommendation Module - Energy Optimization & Rule Engine

**Branch:** `feature/recommendation-module`  
**Input:** `clustering/outputs/energy_cluster_results.csv`  
**Outputs:** `clustering/outputs/recommendations.json`, `clustering/docs/recommendation_rules.md`  

### Objective:
Convert machine learning clustering results into actionable, domain-specific energy optimization recommendations and structured dashboard outputs.

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set plot styles
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## Step 1: Load and Understand Clustered Dataset

In [ ]:
data_path = '../outputs/energy_cluster_results.csv'
df = pd.read_csv(data_path)
df['Total_Power'] = df['Zone_1_Power_Consumption'] + df['Zone_2_Power_Consumption'] + df['Zone_3_Power_Consumption']

print(f"Dataset Shape: {df.shape}")
print("Cluster Counts:")
print(df['Cluster'].value_counts())
df.head()

## Step 2: Compute Cluster Profiles & Statistical Aggregations

In [ ]:
cluster_stats = []
for c in sorted(df['Cluster'].unique()):
    cdf = df[df['Cluster'] == c]
    stats = {
        'Cluster': c,
        'Count': len(cdf),
        'Share (%)': round(len(cdf) / len(df) * 100, 2),
        'Total Power Mean (kW)': round(cdf['Total_Power'].mean(), 2),
        'Zone 1 Mean (kW)': round(cdf['Zone_1_Power_Consumption'].mean(), 2),
        'Zone 2 Mean (kW)': round(cdf['Zone_2_Power_Consumption'].mean(), 2),
        'Zone 3 Mean (kW)': round(cdf['Zone_3_Power_Consumption'].mean(), 2),
        'Temp Mean (°C)': round(cdf['Temperature'].mean(), 2),
        'Humidity Mean (%)': round(cdf['Humidity'].mean(), 2),
        'Peak Hour (%)': round(cdf['Is_Peak_Hour'].mean() * 100, 2),
        'Weekend (%)': round(cdf['Is_Weekend'].mean() * 100, 2),
        'Hour Mean': round(cdf['Hour'].mean(), 2)
    }
    cluster_stats.append(stats)

summary_df = pd.DataFrame(cluster_stats)
display(summary_df)

## Step 3: Visualize Cluster Patterns for Recommendation Design

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Total Power Mean per Cluster
sns.barplot(data=summary_df, x='Cluster', y='Total Power Mean (kW)', ax=axes[0, 0], palette='Reds_d')
axes[0, 0].set_title('Mean Total Power Consumption by Cluster')

# Peak Hour % per Cluster
sns.barplot(data=summary_df, x='Cluster', y='Peak Hour (%)', ax=axes[0, 1], palette='Oranges_d')
axes[0, 1].set_title('Peak Hour Occurrence (%) by Cluster')

# Temperature per Cluster
sns.barplot(data=summary_df, x='Cluster', y='Temp Mean (°C)', ax=axes[1, 0], palette='YlOrRd')
axes[1, 0].set_title('Mean Temperature (°C) by Cluster')

# Zone breakdown per Cluster
zone_df = summary_df.melt(id_vars=['Cluster'], value_vars=['Zone 1 Mean (kW)', 'Zone 2 Mean (kW)', 'Zone 3 Mean (kW)'], var_name='Zone', value_name='Power')
sns.barplot(data=zone_df, x='Cluster', y='Power', hue='Zone', ax=axes[1, 1], palette='Blues_d')
axes[1, 1].set_title('Zone Consumption Breakdown by Cluster')

plt.tight_layout()
plt.show()

## Step 4: Generate Rule Mapping & Export `recommendations.json`

In [ ]:
output_json_path = '../outputs/recommendations.json'
os.system('python ../../scratch/build_recommendations.py')
print(f"Checked recommendations.json: {os.path.exists(output_json_path)}")